In [1]:
import pandas as pd

In [2]:
#Importing Data

#Products
url= 'https://drive.google.com/file/d/1afxwDXfl-7cQ_qLwyDitfcCx3u7WMvkU/view?usp=sharing'
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_df = pd.read_csv(path)

#Orders
url= 'https://drive.google.com/file/d/1oHwUzohlDAT161sfzjX5WVQfIZ1ZLQRd/view?usp=drive_link'
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_df = pd.read_csv(path)

#Orderlines
url= 'https://drive.google.com/file/d/1x6BI5Bsb-mLFO-O_gct7WF5UbU4s90Py/view?usp=drive_link'
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_df = pd.read_csv(path)

#Brands
url= 'https://drive.google.com/file/d/1ACHWXEnL9TxNWjrtWpxszV29YrJDNWa2/view?usp=drive_link'
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
brands_df = pd.read_csv(path)

In [3]:
#Creating copies

products= products_df.copy()
orders= orders_df.copy()
orderlines= orderlines_df.copy()
brands= brands_df.copy()

# Cleaning Dataframe PRODUCTS:

In [4]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19326 entries, 0 to 19325
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   sku          19326 non-null  object
 1   name         19326 non-null  object
 2   desc         19319 non-null  object
 3   price        19280 non-null  object
 4   promo_price  19326 non-null  object
 5   in_stock     19326 non-null  int64 
 6   type         19276 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.0+ MB


In [5]:
#Dropping duplicates

products.duplicated().value_counts(normalize= True)
products= products.drop_duplicates()

In [6]:
products['sku'].nunique()           #10579
len(products['sku'])                #10580

10580

We still have any duplicate row for same product. So we need to find it out and drop.


In [7]:
#Let's check first

products.loc[products.duplicated(subset=['sku'],keep=False)]

,sku,name,desc,price,promo_price,in_stock,type
7992,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,1729,1305.59,0,1282
8000,APP1197,"Apple iMac 21.5 ""Core i5 31 GHz Retina display...",Desktop Apple iMac 21.5 inch i5 31 GHz Retina ...,NaN,1305.59,0,1282


In [8]:
products.drop_duplicates(subset='sku',inplace=True)

## Dealing with Null values
1. Null values in desc.

We gonna use description in further analyses so let's fill its null values with anything.

In [9]:
products.loc[products_df['desc'].isna()]

#Fill it with name
products.loc[products['desc'].isna(),'desc'] = products.loc[products['desc'].isna(),'name']

In [10]:
#% of Null values in type
(products['type'].isna().value_counts(normalize= True))*100


,proportion
type,
False,99.527366
True,0.472634


2. Null values in price.

In [ ]:
#% of Null values in price
(products['price'].isna().value_counts(normalize= True))*100


,proportion
price,
False,99.574629
True,0.425371


Column price has 46 null values out of which I guess we can still fill some using orderlines dataframe. Using SKU we can find the Max price of the product at which it has been sold.

In [ ]:
products_df['sku'].nunique()

10579

In [ ]:
#Filling the missing values of price

max_sku_prices = orderlines_df.groupby('sku')['unit_price'].max().reset_index()
merged_df= products.merge(max_sku_prices, on= 'sku', how= 'left')
merged_df.loc[merged_df['price'].isna(),'price']= merged_df.loc[merged_df['price'].isna(),'unit_price']
products= merged_df.drop(columns= 'unit_price')

In [ ]:
products['sku'].nunique()

10579

In [ ]:
products.dropna(subset= ['price'], inplace=True)

We still have some null values in column 'type' but as per my understanding we are not gonna use that column further in our analyses. So, for now ignoring it and not droping rows (as they have other data) is a good idea.

Now we are left with 22 null values.  In my opinion, we can drop them safely, as we have seen that they were not ordered anyhow.

## Changing type of data.


In [ ]:
#Convert column 'price' into data type numeric

#% of corrupted values
products['price'].str.count(r'\.').value_counts(normalize= True)

#Dropping the corrupted values
p2= products.loc[products['price'].str.count('\.')== 2].index

products= products.drop(p2)

#Converting 'price' to numeric
products['price']= pd.to_numeric(products['price'], errors='coerce')

In [ ]:
#products.loc[p2,['price','promo_price','sku']].nlargest(15,'price')

1. First idea of cleaning and sorting column Promo_Price:

In [ ]:
# #Changing data type of column Promo_Price from object to numeric.
# #It has few values that have two decimel points so using simple 'to_numeric' will not work here. We have to get rid of one decimel point first.

# products['promo_price'].str.count('\.').value_counts()

# #promo_price with zero decimel
# pp0= products.loc[products['promo_price'].str.count('\.')== 0, ['price', 'promo_price']]

# #Promo_Price with 1 decimel.
# pp1= products.loc[products['promo_price'].str.count('\.')== 1].index

# #Moving a decimel point to one digit left.
# products.loc[pp1,'promo_price']= products.loc[pp1,'promo_price'].astype(str).str.replace(r'(\d+)(\.)(\d+)', lambda m: m.group(1)[:len(m.group(1))-1] + '.' + m.group(1)[-1] + m.group(3), regex=True)

# #Promo_Price with 2 decimel points.
# pp2= products.loc[products['promo_price'].str.count('\.')== 2].index

# #Removing first decimal point.
# products.loc[pp2,'promo_price']= products.loc[pp2,'promo_price'].astype(str).str.replace(r'(\d+)\.(\d+)\.(\d+)', lambda m: m.group(1) + m.group(2) + '.' + m.group(3), regex=True)

# #Moving second decimel point to one digit left.
# products.loc[pp2,'promo_price']= products.loc[pp2,'promo_price'].astype(str).str.replace(r'(\d+)(\.)(\d+)', lambda m: m.group(1)[:len(m.group(1))-1] + '.' + m.group(1)[-1] + m.group(3), regex=True)

# #Converting 'promo_price' to numeric
# products['promo_price'] = round(pd.to_numeric(products['promo_price']),2)

# #Check
# (products.loc [round(products['promo_price']) > round(products['price']) , [('price'), ('promo_price')]].count() / products.shape[0])*100

2. Second idea of cleaning and sorting column Promo_Price:

In [ ]:
#Removing first dot from values having 2 decimel points
pp2= products.loc[products['promo_price'].str.count('\.')== 2].index
products.loc[pp2,'promo_price']= products.loc[pp2,'promo_price'].astype(str).str.replace(r'(\d+)\.(\d+)\.(\d+)', lambda m: m.group(1) + m.group(2) + '.' + m.group(3), regex=True)

#Converting 'promo_price' to numeric
products['promo_price']= pd.to_numeric(products['promo_price'], errors='coerce')

#Filtering out the corrupted values
mask= products.loc [round(products['promo_price']) > round(products['price'])].index

#Correcting the position of decimel by moving it to one digit left.
products.loc[mask,'promo_price']= round((products.loc[mask,'promo_price']/10),2)

# #Check
# ((products.loc[round(products['promo_price']) > round(products['price'])]).count()/ len(products['promo_price']))*100

# #% of still crupted values
# 1.21

# #To get rid of them
# still_crupted= (products.loc[round(products['promo_price']) > round(products['price'])]).index
# products= products.drop(still_crupted)

Second idea makes more sense and all the promo prices are now either less or equall to orignal price.


***I sorted data in promo_price (using 'Second idea of cleaning and sorting column Promo_Price') but skipped dropping rows based on corrupted values in promo_price because we probably not be using this column in our further analyses. So, losing data based on it is pointless.***

In [ ]:
products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10180 entries, 0 to 10578
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sku          10180 non-null  object 
 1   name         10180 non-null  object 
 2   desc         10180 non-null  object 
 3   price        10180 non-null  float64
 4   promo_price  10180 non-null  float64
 5   in_stock     10180 non-null  int64  
 6   type         10133 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 894.3+ KB


# Cleaning Dataframe ORDERS:

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   order_id      226909 non-null  int64  
 1   created_date  226909 non-null  object 
 2   total_paid    226904 non-null  float64
 3   state         226909 non-null  object 
dtypes: float64(1), int64(1), object(2)
memory usage: 6.9+ MB


In [ ]:
#Duplicates

orders.duplicated().sum()              #has no duplicates

0

In [ ]:
#Droping Null values

#% of Null values
orders['total_paid'].isna().value_counts(normalize=True)

#drop Null values
orders = orders.dropna(axis=0)   #only column 'total_paid' has null values

In [ ]:
#Correcting the datatypes

#created_date should become datetime datatype
orders["created_date"] = pd.to_datetime(orders["created_date"])

<ipython-input-69-12e5f36122a9>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders["created_date"] = pd.to_datetime(orders["created_date"])


In [ ]:
orders['total_paid'].loc[orders['total_paid']> 1000]

,total_paid
18,1610.00
31,1367.11
35,2264.60
40,1132.33
46,3109.57
...,...
226793,1165.99
226828,3503.99
226831,3075.00
226849,1335.99


In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


# Cleaning Dataframe ORDERLINES:

In [ ]:
orderlines.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293983 entries, 0 to 293982
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   id                293983 non-null  int64 
 1   id_order          293983 non-null  int64 
 2   product_id        293983 non-null  int64 
 3   product_quantity  293983 non-null  int64 
 4   sku               293983 non-null  object
 5   unit_price        293983 non-null  object
 6   date              293983 non-null  object
dtypes: int64(4), object(3)
memory usage: 15.7+ MB


In [ ]:
#Duplicates

orderlines.duplicated().sum()              #has no duplicates

0

In [ ]:
#Correcting the datatypes

#date should be a datetime datatype
orderlines["date"] = pd.to_datetime(orderlines["date"])

#unit_price should be a float datatype

# Count the number of decimal points in the unit_price
orderlines['unit_price'].str.count("\.").value_counts()

#It is string because it has few values that have two decimal points.

#Let's work out how much that is as a percentage of our total data.
mult_decimal_rows = (orderlines['unit_price'].str.count("\.")>1).value_counts(normalize= True)                             #12.3%

In [ ]:
mult_decimal_rows

,proportion
unit_price,
False,0.876969
True,0.123031


In [ ]:
# #Let's drop these 12.3%

# two_dot_order_ids_list = orderlines_df.loc[orderlines_df.unit_price.str.contains("\d+\.\d+\.\d+"), "id_order"]

# orderlines_df = orderlines_df.loc[~orderlines_df.id_order.isin(two_dot_order_ids_list)]

We'll try to save the corrupted values by comparing 'unit_price' from orderlines to 'price' from products.


In [ ]:
#Boolean mask to find the orders that contain a price with multiple decimal points
multiple_decimal_mask = orderlines['unit_price'].str.count("\.") > 1

# Apply the boolean mask to the orderlines DataFrame. This way we can find the order_id of all the affected orders.
corrupted_order_ids = orderlines.loc[multiple_decimal_mask,:]

#Merge the dataframes
comparing= corrupted_order_ids.merge(products, on= 'sku', how= 'left')
comparing[['id_order','sku','price','unit_price']]

,id_order,sku,price,unit_price
0,299544,APP1582,1219.00,1.137.99
1,299549,PAC0929,3209.00,2.565.99
2,299553,APP1854,3279.00,3.278.99
3,299582,PAC0961,2929.00,2.616.99
4,299596,PAC1599,3289.00,2.873.99
...,...,...,...,...
36164,452946,APP2075,3305.59,2.999.00
36165,527321,PAC2148,4519.00,3.497.00
36166,527324,PAC2117,3319.00,3.075.00
36167,527342,APP2492,NaN,1.329.00


By comparison we can see that this first decimal point is somehow read wrong. Unit_price 1.137.99 is actually could be 1137.99 because its 'orignal price' in products is 1219.00. Same for other values.

In [ ]:
#SO, let's remove the first decimal from unit_price and save the data :)

#Removing first dot from values having 2 decimel points
pp2= orderlines.loc[orderlines['unit_price'].str.count('\.')== 2].index
orderlines.loc[pp2,'unit_price']= orderlines.loc[pp2,'unit_price'].astype(str).str.replace(r'(\d+)\.(\d+)\.(\d+)', lambda m: m.group(1) + m.group(2) + '.' + m.group(3), regex=True)

#Change 'unit_price' to numeric
orderlines['unit_price']= pd.to_numeric(orderlines['unit_price'], errors= 'coerce')

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 226904 entries, 0 to 226908
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   order_id      226904 non-null  int64         
 1   created_date  226904 non-null  datetime64[ns]
 2   total_paid    226904 non-null  float64       
 3   state         226904 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


## Saving the clean dataframes:

In [ ]:
from google.colab import files

#Clean dataframe of Products:
products.to_csv('products_cl.csv', index=False)
files.download('products_cl.csv')

#Clean dataframe of Orders
orders.to_csv('orders_cl.csv', index=False)
files.download('orders_cl.csv')

#Clean dataframe of Orderlines
orderlines.to_csv('orderlines_cl.csv', index=False)
files.download('orderlines_cl.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>